In [70]:
import pandas as pd
import numpy as np
import igraph
import pickle as pk
import os
import pycountry_convert as pc

In [71]:
print("Available datasets:")
for d in set( [d.split('_')[0] for d in os.listdir("data/")] ):
    print("-", d)

Available datasets:
- LitLangMed
- Science
- Humanities
- author
- .DS


In [72]:
dataset = 'LitLangMed'
explode = True

In [73]:
print ("data/" + dataset + "_llm_annotated.csv")
#NOTE: Keeping na values for cells labels as n/a, null, etc
df = pd.read_csv( "data/" + dataset + "_llm_annotated.csv", dtype={43:str, 44:str, 45:str, 46:str}, keep_default_na=False, na_values=[''])
#df = pd.read_csv( "data/" + dataset + "_sample3-150.csv", dtype={43:str, 44:str, 45:str, 46:str}, keep_default_na=False, na_values=[''])
print( df.shape )
print( df.columns )

data/LitLangMed_llm_annotated.csv
(7017, 64)
Index(['School', 'Folder', 'Document Name', 'Thesis Title', 'Thesis Author',
       'Citation ID', 'Citation Type', 'Citation Title', 'Citation Authors',
       'Location in thesis', 'Function of ref', 'Quality of ref',
       'Number occurance', 'Citation Editors', 'Citation Year', 'Journal',
       'Publisher', 'Citation Source', 'Place', 'Series', 'Volume', 'Issue',
       'Pages', 'DOI', 'Reference URL', 'Note', '_is_duplicate_within_thesis',
       '_is_noise', '_noise_reason', '_citation_type_original',
       'eligible_for_matching', '_ineligible_reason', 'Thesis Year',
       'Citation Type (refined)', 'Citation Author Count',
       'Citation Is Single Author', 'ref_id', 'openalex_status',
       'openalex_affiliations', 'openalex_countries',
       'openalex_author_breakdown', 'openalex_author_country_pairs',
       'openalex_matched_author_count', 'openalex_main_author_matched',
       'openalex_distinct_author_countries',
       

In [74]:
# Drop rows with nan authors
df = df.dropna(subset=['Citation Authors'])
len(df)

6436

# Get author continents

In [75]:
def get_continent_name(country_code):
    try:
        # Convert 3-letter code to 2-letter code if necessary
        if len(country_code) == 3:
            country_code = pc.country_alpha3_to_country_alpha2(country_code)
            
        # Get the 2-letter continent code (e.g., 'AF', 'NA', 'EU')
        continent_code = pc.country_alpha2_to_continent_code(country_code.upper())
        
        # Map the continent code to its full name
        continent_name = pc.convert_continent_code_to_continent_name(continent_code)
        return continent_name
    except KeyError:
        return "Unknown"

In [76]:
continents = []

for index, row in df.iterrows():
    
    if row['openalex_countries'] is np.nan:
        continents.append(np.nan)
        continue
    
    author_continents = set()
    
    for country in row['openalex_countries'].split(';'):
        country = country.strip()
        author_continents.add( get_continent_name( country ) )

    if 'Africa' in author_continents:
        continents.append( 'Africa' )
    else:
        continents.append( 'Non-Africa' )

df['continent'] = continents
df['continent'] = df['continent'].fillna('Unknown')

df['continent'].value_counts()

continent
Unknown       3531
Non-Africa    2276
Africa         629
Name: count, dtype: int64

## Explode the dataframe

In [77]:
if explode:
    df['Citation Authors'] = df['Citation Authors'].str.split(';')
    df = df.explode('Citation Authors', ignore_index=True)
    len(df)

10303

# Extract annotation counts by author

In [78]:
# # Variant -> unified label. Keys are in cleaned form (lowercase, underscores).
# # Mapping a key to None drops it. Any label not listed here is kept as is.
# LABEL_MAP = {
#     # --- Location in thesis ---
#     'intro': 'introduction',
#     'introudction': 'introduction',            # typo
#     'lit_rev': 'lit_review',                   # 'LIT REV', 'LIT-REV'
#     'theoretical_framework': 'theory_framework',
#     'conclusions': 'conclusion',
#     'bib': 'bibliography',
#     'not_a_citation': None,
#     # --- Function of ref ---
#     'empicial': 'empirical',                   # typo
#     'verify_findigs': 'verify_findings',       # typo
#     'methodological': 'methodology',
#     # --- Placeholders for missing values (all columns) ---
#     'nul': 'bib', 'null': 'bib', 'n_a': 'bib', 'na': 'bib', 'nan': 'bib', 'none': 'bib,
# }

# SEP = r'[;,:]'     # every separator found in the data


def clean_label(s):
    # 'LIT-REV ' -> 'lit_rev'. This handles case, extra spaces and punctuation.
    return (s.str.lower()
             .str.replace(r'[^a-z0-9]+', '_', regex=True)
             .str.strip('_'))

In [79]:
def count_types(df, col, prefix, author_col='Citation Authors', split_authors = False):
    #loads label mapping and sep each time in case changes
    #label_map = LABEL_MAP
    sep=';'
    
    # Keep authors column and the column being counted.
    # row_id remembers which original row each piece came from.
    tmp = df[[author_col, col]].copy()
    tmp['row_id'] = tmp.index

    if split_authors == False: # baseline case - dont split on authors when co-authorship
        tmp['Authors'] = (tmp[author_col]
                            .str.replace(r'\s*;[\s;]*', '; ', regex=True) # Standardise spacing around ';' and remove empty segments
                            .str.strip('; ')) #drop any extra spaces
    #else can split on authors if we choose
    else:
        #Split on ';' so each author gets their own row
        tmp['Authors'] = tmp[author_col].str.split(';') # split (spaces stripped below)
        tmp = tmp.explode('Authors')
        tmp['Authors'] = tmp['Authors'].str.strip()
    
    # Get all unique author list for total node set
    all_authors = tmp['Authors'].dropna()
    all_authors = all_authors[all_authors != ''].unique()
    # Drop blank authors
    tmp = tmp.dropna(subset=['Authors'])
    tmp = tmp[tmp['Authors'] != '']
    
    # One row per label: split on any separator in `sep`, then drop blank cells.
    tmp['type'] = tmp[col].astype('string').str.split(sep, regex=True)
    tmp = tmp.explode('type').dropna(subset=['type'])

    ## Assume clean labels, code kepts for use on unclean labels
    ## Clean each label, apply the mapping, then drop empty or None labels
    tmp['type'] = clean_label(tmp['type'])
    #tmp['type'] = tmp['type'].map(lambda t: label_map.get(t, t))
    #tmp = tmp[tmp['type'].notna() & (tmp['type'] != '')]

    # Count each label once per cell, then add the column prefix
    tmp = tmp.drop_duplicates(subset=['row_id', 'Authors', 'type'])
    tmp['type'] = prefix + '_' + tmp['type']

    # Count rows per (author, type), then pivot the types into columns
    # Duplicate authors are merged here. Authors with no types get 0s
    counts = (tmp.groupby(['Authors', 'type']).size()
                .unstack(fill_value=0)
                .reindex(sorted(all_authors), fill_value=0)
                .rename_axis(index='Authors', columns=None)
                .reset_index())
    return counts

# to sum over all occourence counts (counts generated per thesis citation)
def sum_occurrences(df, col='Number occurance', author_col='Citation Authors', split_authors=False):
    tmp = df[[author_col, col]].copy()
    tmp[col] = pd.to_numeric(tmp[col], errors='coerce').fillna(0)   # text such as 'nul' or 'AMBIGUOUS' -> 0
    if split_authors == False:
        tmp['Authors'] = (tmp[author_col]
                            .str.replace(r'\s*;[\s;]*', '; ', regex=True)
                            .str.strip('; '))
    else:
        tmp['Authors'] = tmp[author_col].str.split(';')
        tmp = tmp.explode('Authors')
        tmp['Authors'] = tmp['Authors'].str.strip()
    tmp = tmp[tmp['Authors'].notna() & (tmp['Authors'] != '')]
    return (tmp.groupby('Authors')[col].sum().astype(int)
               .rename('total_occurrences').reset_index())


SPLIT = False   # set once, so every table uses the same author mode
loc_counts  = count_types(df, 'Location in thesis', 'loc', split_authors=SPLIT)
func_counts = count_types(df, 'Function of ref', 'func', split_authors=SPLIT)
qual_counts = count_types(df, 'Quality of ref', 'qual', split_authors=SPLIT)
occ_counts  = sum_occurrences(df, split_authors=SPLIT)
author_counts = (loc_counts.merge(func_counts, on='Authors')
                    .merge(qual_counts, on='Authors')
                    .merge(occ_counts, on='Authors', how='left'))

In [80]:
author_counts.head()

,Authors,loc_ambiguous,loc_analysis,loc_bib,loc_conclusion,loc_introduction,loc_limitation,loc_lit_review,loc_methodology,loc_preface,loc_theory_framework,func_empirical,func_methods,func_n_a,func_theory,qual_n_a,qual_passing,qual_summary,total_occurrences
0,& Brothers,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
1,& Neill,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,& Pinch,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,1,0,0
3,& Wishart,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,1,0,6
4,' Gazette,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,4


In [81]:
#write to csv to check
if explode:
    author_counts.to_csv('data/author_counts_exploded.csv', index=False, encoding='utf-8-sig')
else:
    author_counts.to_csv('data/author_counts.csv', index=False, encoding='utf-8-sig')

### Merge author counts to citation data and aggregate

In [82]:
df = pd.merge(df, author_counts, how="left", left_on="Citation Authors", right_on="Authors")
df = df.drop(columns=['Authors'])

In [83]:
# Get dominant categories
df['loc_dominant'] = df.filter(regex='^loc_').fillna(0).idxmax(axis=1).str.removeprefix('loc_')
df['func_dominant'] = df.filter(regex='^func_').fillna(0).idxmax(axis=1).str.removeprefix('func_')
df['qual_dominant'] = df.filter(regex='^qual_').fillna(0).idxmax(axis=1).str.removeprefix('qual_')
df

,School,Folder,Document Name,Thesis Title,Thesis Author,Citation ID,Citation Type,Citation Title,Citation Authors,Location in thesis,...,func_methods,func_n_a,func_theory,qual_n_a,qual_passing,qual_summary,total_occurrences,loc_dominant,func_dominant,qual_dominant
0,"Literature, Language and Media",1,Mnguni_Analyzing_2024.pdf,Analyzing Financial Survival Strategies for Pu...,Thamba Mnguni; Yolo Koba,1,Web/Grey Literature,Rethinking Public Service Broadcasting's Place...,S Abbott,LIT REVIEW,...,0.0,1.0,0.0,0.0,1.0,0.0,1.0,lit_review,n_a,passing
1,"Literature, Language and Media",1,Mnguni_Analyzing_2024.pdf,Analyzing Financial Survival Strategies for Pu...,Thamba Mnguni; Yolo Koba,2,Book Chapter,Media and Cultural Industries,T Adorno,LIT REVIEW,...,0.0,0.0,1.0,0.0,0.0,2.0,2.0,analysis,empirical,summary
2,"Literature, Language and Media",1,Mnguni_Analyzing_2024.pdf,Analyzing Financial Survival Strategies for Pu...,Thamba Mnguni; Yolo Koba,2,Book Chapter,Media and Cultural Industries,M Horkheimer,LIT REVIEW,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ambiguous,empirical,n_a
3,"Literature, Language and Media",1,Mnguni_Analyzing_2024.pdf,Analyzing Financial Survival Strategies for Pu...,Thamba Mnguni; Yolo Koba,3,Web/Grey Literature,Load shedding gives rise to growth in SA Strea...,L Adams,LIT REVIEW,...,0.0,1.0,0.0,0.0,0.0,1.0,1.0,lit_review,n_a,summary
4,"Literature, Language and Media",1,Mnguni_Analyzing_2024.pdf,Analyzing Financial Survival Strategies for Pu...,Thamba Mnguni; Yolo Koba,4,Journal Article,Disability and sexuality: Claiming sexual and ...,R Addlakha,THEORY FRAMEWORK,...,0.0,0.0,0.0,0.0,0.0,1.0,1.0,theory_framework,empirical,summary
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10298,"Literature, Language and Media",15,Olaniran_2022Disinformation_2022.pdf,NaN,NaN,683,Journal Article,"Tweet, but verify: Epistemic study of informat...",A Zubiaga,CONCLUSION,...,0.0,0.0,0.0,0.0,1.0,0.0,1.0,conclusion,empirical,passing
10299,"Literature, Language and Media",15,Olaniran_2022Disinformation_2022.pdf,NaN,NaN,683,Journal Article,"Tweet, but verify: Epistemic study of informat...",H Ji,CONCLUSION,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ambiguous,empirical,n_a
10300,"Literature, Language and Media",15,Olaniran_2022Disinformation_2022.pdf,NaN,NaN,684,Other,1708153@students.wits.ac,Samuel Yours Sincerely,BIB,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,bib,empirical,n_a
10301,"Literature, Language and Media",15,Olaniran_2022Disinformation_2022.pdf,NaN,NaN,684,Other,1708153@students.wits.ac,Olaniran,BIB,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ambiguous,empirical,n_a


# Create igraph network

In [84]:
def create_graph_data(df):
    
    ## CREATE NODES

    # Get unique phd documents based on doc name
    df_phds = df.drop_duplicates(subset=['Document Name'])[['Document Name', 'School']]
    df_phds.columns = ['title', 'school']
    df_phds['type'] = 'Wits PhD'

    # Get unique refs based on title, authors and ref type
    df_refs = df.drop_duplicates(subset=['Citation Authors'])[
        ['Citation Authors', 'Citation Type', 'continent', 'loc_dominant', 'func_dominant', 'qual_dominant']
        ]
    df_refs.columns = ['authors', 'type', 'continent', 'location', 'function', 'quality']

    # Concatenate into single node df and generate unique id for each node
    df_nodes = pd.concat( [
        df_phds, 
        df_refs
    ], ignore_index = True).reset_index(names='id')

    
    ## CREATE EDGES

    # Create mappings from phd docs and refs to node ids
    phd_map = df_nodes[df_nodes['type'] == 'Wits PhD'].set_index('title')['id']    
    ref_map = df_nodes[df_nodes['type'] != 'Wits PhD'].set_index(['authors'])['id']

    # Create edges by mapping phd doc names into source column, and ref title, author and data into target column
    df_edges = pd.DataFrame({
        'source': df['Document Name'].map(phd_map),
        'target': df['Citation Authors'].map(ref_map)
    })

    return df_nodes, df_edges

def describe(g):
    print( g.vcount() )
    print( g.ecount() )
    print( g.vertex_attributes() )
    print( g.edge_attributes() )

In [85]:
nodes, edges = create_graph_data(df)

In [89]:
edges = edges.drop_duplicates().reset_index(drop=True)

In [90]:
g = igraph.Graph.DataFrame(edges, directed=True, vertices=nodes, use_vids=False)
describe(g)

8050
8548
['name', 'title', 'school', 'type', 'authors', 'continent', 'location', 'function', 'quality']
[]


In [91]:
## Write graph to pickle
if explode:
    g.write_pickle( "graphs/" + dataset + "_graph_exploded.pk")
else:
    g.write_pickle( "graphs/" + dataset + "_graph.pk")

## Plot cit network for presentation

In [66]:
from fa2_modified import ForceAtlas2
import random

In [67]:
random.seed(1327)

forceatlas2 = ForceAtlas2(
                        # Behavior alternatives
                        outboundAttractionDistribution=False,  # Dissuade hubs
                        linLogMode=False,  # NOT IMPLEMENTED
                        adjustSizes=False,  # Prevent overlap (NOT IMPLEMENTED)
                        edgeWeightInfluence=1,

                        # Performance
                        jitterTolerance=1.0,  # Tolerance
                        barnesHutOptimize=True,
                        barnesHutTheta=1.2,
                        multiThreaded=False,  # NOT IMPLEMENTED

                        # Tuning
                        scalingRatio=1.0,
                        strongGravityMode=True,
                        gravity=1,

                        # Log
                        verbose=True)

l = forceatlas2.forceatlas2_igraph_layout( g, pos=None, iterations=100 )

100%|█████████████████████████████████████████| 100/100 [00:08<00:00, 11.24it/s]

BarnesHut Approximation  took  1.99  seconds
Repulsion forces  took  6.64  seconds
Gravitational forces  took  0.00  seconds
Attraction forces  took  0.01  seconds
AdjustSpeedAndApplyForces step  took  0.12  seconds


In [69]:
igraph.plot(
    g,
    'plots/' + dataset + "_" + 'citation_network.pdf',
    layout=l, 
    vertex_size = 1,
    edge_width=0.1,
    bbox=(1000,1000),
    margin=20
)